# 07c — Explainability and Nonlinear Structure

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
from scorelab import tuning as T, explain as E, survival as SV, causal as CA
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

What the best challenger learned that the card cannot carry, in a form a validator can read: SHAP summary and dependence plots and partial dependence for its top attributes (with the card's own shape beside each), how much of the model is interaction (pairwise SHAP interaction values and Friedman's H), which interactions can be carried back into the card as crossed attributes and what Gini they recover, and whether the card and the GBM give the same adverse-action reasons to the same declined applicants.

Decisions: **Y — which interactions to carry back into the card**, **Z — the reason-code position** (are GBM reasons acceptable if the GBM were promoted?).

In [ ]:
frame = load_frame()
tr, va, oot = (frame[frame.split == s] for s in ("train", "validate", "oot"))
card, specs = S.load_card(MODELS / "champion"); card_attrs = card.attributes
cands = json.loads((MODELS / "candidate_attributes.json").read_text(encoding="utf-8"))["candidates"]
print(f"train {len(tr):,} · validate {len(va):,} · oot {len(oot):,} · {len(cands)} candidates · card on {len(card_attrs)}")

import shap
booster = C.load_xgb(MODELS / "challenger_xgb_tuned.json"); g_attrs = booster.attributes_used
p_oot = C.predict_p(booster, oot)
print("challenger OOT gini %.4f vs card %.4f" % (V.auc_gini_ks(oot["bad"], p_oot, higher_is_safer=False)["gini"],
                                               V.auc_gini_ks(oot["bad"], S.score(oot, card, specs)["score"])["gini"]))

In [ ]:
# Evidence — SHAP summary: which attributes drive the challenger, and in which direction
sv, bias = C.shap_values(booster, oot, max_rows=20000)
imp = C.shap_importance(sv); display(imp.head(15))
TOP = imp.attribute.head(8).tolist()
Xs = oot.loc[sv.index, g_attrs]
shap.summary_plot(sv[TOP].to_numpy(), Xs[TOP], show=False, max_display=len(TOP)); plt.title("SHAP summary (log-odds of bad), 2016"); plt.show()

In [ ]:
# Evidence — SHAP dependence for the top 4, coloured by the attribute each interacts with most
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for a, ax in zip(TOP[:4], axes.ravel()):
    shap.dependence_plot(a, sv.to_numpy(), Xs, interaction_index="auto", ax=ax, show=False)
plt.tight_layout(); plt.show()

In [ ]:
# Evidence — partial dependence (with ICE) against the card's own shape for the same attribute
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for a, ax in zip(TOP[:6], axes.ravel()):
    pdt, ice = E.partial_dependence(booster, oot, a, n_grid=20, max_rows=3000, ice=40)
    num = pdt[~pdt.is_missing]
    ax.plot(ice.index[:len(num)], ice.iloc[:len(num)] - ice.iloc[:len(num)].mean(), c="lightgrey", lw=0.6)
    ax.plot(num.value, num.mean_logit - num.mean_logit.mean(), c="C0", lw=2.5, label="challenger PD")
    if a in card_attrs:
        w = B.transform_woe(pd.DataFrame({a: num.value.to_numpy()}), {a: specs[a]})[a + "_woe"] * card.coef[a]
        ax.step(num.value, w - w.mean(), where="post", c="C3", lw=2, label="card (beta x WOE)")
    if pdt.is_missing.any():
        ax.axhline(pdt.mean_logit.iloc[-1] - num.mean_logit.mean(), ls=":", c="C0", label="challenger, missing")
    ax.set_title(a); ax.set_ylabel("centred log-odds of bad"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Evidence — interaction strength: exact pairwise SHAP interactions, then Friedman's H on the top pairs
pairs = E.shap_interaction_pairs(booster, oot, top=15, max_rows=5000); display(pairs)
H = E.friedman_h_pairs(booster, oot, list(zip(pairs.a, pairs.b))[:8], n_rows=300); display(H)
inter = C.interaction_strength(booster, g_attrs, oot, max_rows=3000)
print("share of SHAP variance that is interaction, mean over attributes: %.3f" % inter.interaction_share.mean())
pairs.merge(H, on=["a", "b"], how="left").to_csv(config.DOCS_DIR / "exhibits" / "interaction_pairs.csv", index=False)

### Decision Y — Interactions to carry back

A pair is worth carrying back when it is strong on both measures (SHAP interaction mass and H²), the crossed attribute has a story you can tell a consumer ("high utilisation *and* a recent inquiry"), and it is stable across vintages. Each candidate below is crossed at the GBM's own best split thresholds, binned like any categorical attribute, and offered to the stepwise selection with the card's attributes; the Gini gain is the paired DeLong difference on 2016. Pick the pairs to keep (edit `CARRY_BACK`).

**Your reasoning:**

_(write here)_

In [ ]:
CARRY_BACK = list(zip(pairs.a, pairs.b))[:3]          # edit
rows, extra = [], {}
tr_x, oot_x = tr.copy(), oot.copy()
for a, b in CARRY_BACK:
    ta, tb = E.top_threshold(booster, a), E.top_threshold(booster, b)
    col = f"{a}_x_{b}"
    tr_x[col] = E.cross_attribute(tr, a, b, ta, tb); oot_x[col] = E.cross_attribute(oot, a, b, ta, tb)
    spec = B.fit_categorical(tr_x[col], tr_x["bad"], name=col)
    r = E.refit_card_with(tr_x, oot_x, card, specs, {col: spec})
    rows.append({"pair": col, "thresholds": (round(ta, 3), round(tb, 3)), "iv": round(spec.iv, 4), "entered": bool(r["entered"]),
                 "gini_before": round(r["gini_before"], 4), "gini_after": round(r["gini_after"], 4),
                 "gini_gain": round(r["gini_gain"], 4), "p_value": round(r["p_value"], 4)})
    extra[col] = spec
carry = pd.DataFrame(rows); display(carry)
joint = E.refit_card_with(tr_x, oot_x, card, specs, extra)
print(f"all {len(extra)} crosses offered together: entered {joint['entered']} · Gini {joint['gini_before']:.4f} -> {joint['gini_after']:.4f} (p = {joint['p_value']:.3g})")
carry.to_csv(config.DOCS_DIR / "exhibits" / "interaction_carry_back.csv", index=False)

In [ ]:
# Evidence — reason-code agreement for the applicants the card would decline at the pilot approval rate
sc = S.score(oot, card, specs)["score"]
declined = oot[sc < sc.quantile(1 - config.APPROVAL_RATE_FOR_SWAP)]
sample = declined.sample(min(3000, len(declined)), random_state=42)
rc_card = S.reason_codes(sample, card, specs, meanings=MEANINGS, missing_reasons=A.MISSING_REASONS)
sv_d, _ = C.shap_values(booster, sample, max_rows=len(sample))
rc_gbm = C.gbm_reason_codes(sv_d.loc[sample.index], meanings=MEANINGS)
# Like for like: a GBM restricted to the card's attributes, with the tuned settings
tuning = json.loads((MODELS / "challenger_tuning.json").read_text(encoding="utf-8"))
g_card = C.fit_xgb(tr, tr["bad"], va, va["bad"], card_attrs, monotone=True, params=tuning["xgb_params"])
rc_gbm_card = C.gbm_reason_codes(C.shap_values(g_card, sample, max_rows=len(sample))[0].loc[sample.index], meanings=MEANINGS)
agree = pd.DataFrame({"GBM, all candidates": E.reason_code_agreement(rc_card, rc_gbm)[0],
                      "GBM, card attributes only": E.reason_code_agreement(rc_card, rc_gbm_card)[0]})
display(agree.round(3))
display(E.first_reason_crosstab(rc_card, rc_gbm_card))
agree.to_csv(config.DOCS_DIR / "exhibits" / "reason_code_agreement.csv")

### Decision Z — Reason-code position

The card's reasons are points lost against each attribute's best bin; the GBM's are the largest positive SHAP contributions against the *average* applicant. These are different reference points, so some disagreement is by construction. Is the agreement on the first reason high enough that a consumer would hear the same story? Where they disagree, which is right (look at a few rows)? If the GBM were promoted, would its reasons pass an adverse-action review?

**Your reasoning:**

_(write here)_